In [1]:
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import astropy

import matplotlib.pyplot as plt
from matplotlib.pyplot import figure, show, savefig, close
import matplotlib.font_manager as fm
import itertools
from matplotlib import rcParams
import scienceplots
import pickle

from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.patches as mpatches
from matplotlib.patches import Patch

import os
import xarray as xr
from taurex.cache import OpacityCache, CIACache
OpacityCache().set_opacity_path('./xsec/')
CIACache().set_cia_path('./cia/')

from taurex.temperature import TemperatureFile
from taurex.chemistry import TaurexChemistry
from taurex.chemistry import ConstantGas

from taurex.planet import Planet
from taurex.stellar import BlackbodyStar, PhoenixStar

from taurex.model import EmissionModel, TransmissionModel
from taurex.pressure import SimplePressureProfile

from explor.model import HotSpotPhaseCurveModel

from taurex.contributions import AbsorptionContribution
from taurex.contributions import CIAContribution
from taurex.contributions import RayleighContribution

from matplotlib.lines import Line2D

from taurex.binning import FluxBinner
from binning_funcs import *
from scipy.optimize import curve_fit, brentq

In [2]:
planet_names = ["HD3167","K2141","lhs1478","TOI431","TOI500","TOI561","TOI1416","TOI1807"]
planet_fullnames = ["HD-3167b","K2-141b","LHS-1478b","TOI-431b","TOI-500b","TOI-561b","TOI-1416b","TOI-1807b"]
planet_masses = [4.73, 4.97, 2.33, 3.07, 1.42, 2.02, 3.48, 2.44] #Earth masses
planet_distances = [0.018, 0.007, 0.018, 0.011, 0.012, 0.011, 0.019, 0.012] #AU
planet_period = [0.96, 0.28, 1.95, 0.49, 0.55, 0.45, 1.0, 0.55] #days
planet_radius = [1.627, 1.510, 1.242, 1.277, 1.166, 1.397, 1.620, 1.496] #Earth radii
planet_transit = [1.61, 0.94, 0.71, 1.24, 0.99, 1.31, 1.5, 0.98] #hours
planet_transit = [n * 3600 for n in planet_transit] #hours to seconds
T_transit_hours = [1.61, 0.94, 0.71, 1.24, 0.99, 1.31, 1.5, 0.98] #hours
planet_impact = [0.181, -0.01, 0.717, 0.34, 0.53, 0.14, 0.39, 0.489] #to be fixed according to archive data
planet_eccentricity = [0.05, 0.0, 0.0, 0.0, 0.06, 0.0, 0.0, 0.0] #to be fixed according to archive data
planet_pericentre_long = [0.0, 90.0, 0.0, 0.0, 228.5, 0.0, 0.0, 90.0] #w, to be fixed according to archive data

star_temperature = [5261.0,4570.0,3381.0,4850.0,4440.0,5342.0,4884.0,4914.0] #Kelvin
star_radius = [0.872,0.681,0.246,0.731,0.678,0.856,0.793,0.746] #Solar radii
star_metallicity = [0.03, 0.0, -0.13, 0.2, 0.12, -0.4, 0.08, -0.04] #[Fe/H]
star_logg = [4.5, 4.6, 4.9, 4.6, 4.6, 4.5, 4.5, 4.6]
star_age = [10.2, 6.3, 5.6, 5.1, 5, 11, 6.9, 0.3] #Gyr
star_distance = [47.28, 61.87, 18.22, 32.6, 47.39, 85.8, 55.01, 42.58] #pc, to be fixed

In [3]:
rcParams['xtick.direction'] = 'in'         # ticks pointing inward (standard in astronomy)
rcParams['ytick.direction'] = 'in'
rcParams['xtick.top'] = True               # mirror ticks on top
rcParams['ytick.right'] = True
rcParams['xtick.major.size'] = 5
rcParams['xtick.minor.size'] = 3
rcParams['xtick.minor.visible'] = True
rcParams['ytick.minor.visible'] = True

rcParams['axes.linewidth'] = 2
rcParams['axes.grid'] = True
rcParams['grid.alpha'] = 0.8
rcParams['grid.linestyle'] = '-'

rcParams['font.family'] = 'serif'          # or 'sans-serif' (Nature/AAS style)
rcParams['font.serif'] = ['Times New Roman']
rcParams['font.size'] = 16
rcParams['axes.labelsize'] = 22            # axis label size
rcParams['xtick.labelsize'] = 20
rcParams['ytick.labelsize'] = 20
rcParams['legend.fontsize'] = 16
rcParams['text.usetex'] = True             # render labels with LaTeX

plt.style.use(['science', 'no-latex'])

In [7]:
for name in planet_names:
    for folder in os.listdir(f"PLANETS/{name}/"):
        simulation_folder = os.path.join(f"PLANETS/{name}/", folder)
        #check if it is a directory
        if not os.path.isdir(simulation_folder):
            continue

        #set-up directories and paths
        outputdir = f"PLANETS/{name}/{os.path.basename(simulation_folder)}"
        planetdir = f"PLANETS/{name}/{os.path.basename(simulation_folder)}_TP.csv"

        # open TP profile with TauREx
        temp_profile = TemperatureFile(planetdir, skiprows=1, 
                                temp_col=2, press_col=0, 
                                temp_units='K', press_units='Pa',
                                delimiter = ',')
        
        temp_profile_terminator = TemperatureFile(planetdir, skiprows=1,
                                temp_col=1, press_col=0,
                                temp_units='K', press_units='Pa',
                                delimiter = ',')
        
        #read-in the atmospheric composition
        atm_file = None
        for file in os.listdir(f"PLANETS/{name}/{os.path.basename(simulation_folder)}/"):
            if file.endswith("atm.nc"):
                atm_file = os.path.join(f"PLANETS/{name}/{os.path.basename(simulation_folder)}/", file)
                #save filename without extension
                filename = os.path.splitext(atm_file)[0]
                break

        ds = xr.open_dataset(atm_file)

        #extract gas names
        gases = np.array(ds['gases'])
        gases = [m.decode().strip() for m in ds["gases"].values]
        vmr = np.array(ds['x_gas'])

        pressure = np.array(ds['p']) #pressure in Pa
        pmax = pressure.max()
        pmin = pressure.min()

        radius = float(ds['planet_radius']) #planet radius in m
        #convert to Jupiter radii
        radius = radius / astropy.constants.R_jup.value

        #get mixing ratio of each molecule
        H2O_x = float(vmr[:, gases.index('H2O')][0])
        CO2_x = float(vmr[:, gases.index('CO2')][0])
        CH4_x = float(vmr[:, gases.index('CH4')][0])
        CO_x = float(vmr[:, gases.index('CO')][0])
        NH3_x = float(vmr[:, gases.index('NH3')][0])
        N2_x = float(vmr[:, gases.index('N2')][0])
        SO2_x = float(vmr[:, gases.index('SO2')][0])
        S2_x = float(vmr[:, gases.index('S2')][0])
        O2_x = float(vmr[:, gases.index('O2')][0])
        H2_x = float(vmr[:, gases.index('H2')][0])
        H2S_x = float(vmr[:, gases.index('H2S')][0])

        #re-normalize the mixing ratios so they sum to 1
        total = H2O_x + CO2_x + CH4_x + CO_x + NH3_x + N2_x + SO2_x + S2_x + O2_x + H2_x + H2S_x
        H2O_x /= total
        CO2_x /= total
        CH4_x /= total
        CO_x /= total
        NH3_x /= total
        N2_x /= total
        SO2_x /= total
        S2_x /= total
        O2_x /= total
        H2_x /= total
        H2S_x /= total

        #save in dictionary the planet name, simulation folder, temperature profiles and mixing ratios
        sim_name = os.path.basename(simulation_folder)

        #define chemistry
        chemistry = TaurexChemistry(fill_gases=["N2"])

        chemistry.addGas(ConstantGas(molecule_name="NH3", mix_ratio=NH3_x)).addGas(ConstantGas(molecule_name="CO2", mix_ratio=CO2_x)).addGas(ConstantGas(molecule_name="H2O", mix_ratio=H2O_x)).addGas(ConstantGas(molecule_name="CH4", mix_ratio=CH4_x)).addGas(ConstantGas(molecule_name="CO", mix_ratio=CO_x)).addGas(ConstantGas(molecule_name="SO2", mix_ratio=SO2_x)).addGas(ConstantGas(molecule_name="S2", mix_ratio=S2_x)).addGas(ConstantGas(molecule_name="O2", mix_ratio=O2_x)).addGas(ConstantGas(molecule_name="H2", mix_ratio=H2_x)).addGas(ConstantGas(molecule_name="H2S", mix_ratio=H2S_x))

        mass = planet_masses[planet_names.index(name)] #Earth masses
        #convert to jupiter masses
        mass = mass / 317.8
        #semi-major axis
        a = planet_distances[planet_names.index(name)] #AU
        #set-up planet in Jupiter masses and radii
        planet = Planet(planet_mass=mass, planet_radius= radius, planet_distance=a)

        star = PhoenixStar(temperature=star_temperature[int(planet_names.index(name))], radius=star_radius[int(planet_names.index(name))], metallicity=star_metallicity[int(planet_names.index(name))], phoenix_path='Phoenix/')

        em = EmissionModel(
        planet=planet,
        temperature_profile=temp_profile,
        chemistry=chemistry,
        pressure_profile=SimplePressureProfile(atm_min_pressure=pmin, atm_max_pressure=pmax, nlayers=100),
        star=star,
        )

        em_terminator = EmissionModel( 
        planet=planet,
        temperature_profile=temp_profile_terminator,
        chemistry=chemistry,
        pressure_profile=SimplePressureProfile(atm_min_pressure=pmin, atm_max_pressure=pmax, nlayers=100),
        star=star,
        )

        em.add_contribution(AbsorptionContribution())
        em.add_contribution(CIAContribution(cia_pairs=['CO2-CH4','CO2-CO2','CO2-H2','CO2-H2O','H2-H2','N2-CH4','N2-H2','N2-H2O','N2-N2','O2-CO2','O2-N2','O2-O2']))
        em.add_contribution(RayleighContribution())

        em_terminator.add_contribution(AbsorptionContribution())
        em_terminator.add_contribution(CIAContribution(cia_pairs=['CO2-CH4','CO2-CO2','CO2-H2','CO2-H2O','H2-H2','N2-CH4','N2-H2','N2-H2O','N2-N2','O2-CO2','O2-N2','O2-O2']))
        em_terminator.add_contribution(RayleighContribution())

        em.build()
        em_terminator.build()

        #wavenumber grid, flux ratio, and transit depth
        wngrid, fpfs, tau, _ = em.model()
        wngrid_t, fpfs_t, tau_t, _ = em_terminator.model()

        wlgrid = 10000/wngrid[::-1]; wlgrid_t = 10000/wngrid_t[::-1]
        fpfs = fpfs[::-1]; fpfs_t = fpfs_t[::-1]

        #read-in Ariel noise model
        #check if file exists
        
        if os.path.exists(f"ARIEL/arielrad_{name}/tier1.csv"):
            file = f"ARIEL/arielrad_{name}/tier1.csv"
        else:
            file = f"ARIEL/arielrad_{name}/tier1.csv"

        ariel = pd.read_csv(file,skiprows=6)
        wl = np.array(ariel['Wavelength [um]'])
        wb = np.array(ariel['Bandwidth [um]'])
        noise = np.array(ariel['Noise on Transit Floor [ppm]']) * 1e-6 #converts ppm to fractional
        inst = np.array(ariel['Instrument'])

        #instantiate flux binner with Ariel's wavelength binning
        fb = FluxBinner(wl, wb)
        output = fb.bindown(wlgrid, fpfs)
        output_t = fb.bindown(wlgrid_t, fpfs_t)

        #introduce binning
        binning = False
        #new_point9 = (0.55, 0.1, 0.7, 0.2)
        #new_point10 = (0.95, 0.3, 1.5377, 0.875)

        #new_point11 = (2.3289, 0.757, 3.234, 1.052)
        #new_point12 = (3.779, 0.037, 5.832, 3.864)

        # loop over N until eclipse SNR reaches 5
        for N in range(1, 100):
            if binning:
                new_points = [new_point12]
                results = bindown_multiple(output, noise, wlgrid, fpfs, N, name, *new_points)
                #new_points_level1 = make_next_level_points(results)
                #results_level2 = bindown_multiple(output, noise, wlgrid, fpfs, N, name, *new_points_level1)
                #new_points_level2 = make_next_level_points(results_level2)
                #results_level3 = bindown_multiple(output, noise, wlgrid, fpfs, N, name, *new_points_level2)
            # calculate SNR
                SNR = results[0][3] / (results[0][4])
            else:
                SNR = output[1] / (noise/np.sqrt(N))
                SNR = np.sqrt(np.sum(SNR**2))

            # check if SNR exceeds 5 for any wavelength bin
            if np.any(SNR >= 5):
                print(f"{name}_{simulation_folder}: SNR reaches 5 at N = {N} for wavelength bin(s)")
                break

HD3167_PLANETS/HD3167/H20_IW4_00001: SNR reaches 5 at N = 7 for wavelength bin(s)
HD3167_PLANETS/HD3167/H10_IW4_00001: SNR reaches 5 at N = 6 for wavelength bin(s)
K2141_PLANETS/K2141/H30_IW2_0001_S40: SNR reaches 5 at N = 2 for wavelength bin(s)
K2141_PLANETS/K2141/H20_IW4_0001_S40: SNR reaches 5 at N = 2 for wavelength bin(s)
lhs1478_PLANETS/lhs1478/H05_IW0_0001: SNR reaches 5 at N = 18 for wavelength bin(s)
lhs1478_PLANETS/lhs1478/H05_IW4_0001: SNR reaches 5 at N = 17 for wavelength bin(s)
TOI431_PLANETS/TOI431/H20_IW4_0001_S20: SNR reaches 5 at N = 4 for wavelength bin(s)
TOI431_PLANETS/TOI431/H10_IW4_0001_S40: SNR reaches 5 at N = 4 for wavelength bin(s)
TOI500_PLANETS/TOI500/H20_IW4_00001: SNR reaches 5 at N = 10 for wavelength bin(s)
TOI500_PLANETS/TOI500/H20_IW2_00001: SNR reaches 5 at N = 9 for wavelength bin(s)
TOI561_PLANETS/TOI561/H20_IW4_00001: SNR reaches 5 at N = 17 for wavelength bin(s)
TOI561_PLANETS/TOI561/H20_IW2_00001: SNR reaches 5 at N = 15 for wavelength bin(s)
T

In [ ]:
print(wl, wb)